In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
from numba import njit

sys.path.append(os.path.abspath('./src'))
from ssa_params import monomer_params
params, initial_state, stoichiometry = monomer_params


@njit
def fast_ssa_monomer(alpha_s, beta_s, alpha_n, beta_n, k_y, gamma_y, t_max):
    state = np.array([1.0, 1.0, 0.0, 0.0, 0.0])
    t = 0.0
    
    while t < t_max:
        nf, sf, nb, sb, y = state
        
        p0 = alpha_s * sf
        p1 = beta_s * sb
        p2 = alpha_n * nf
        p3 = beta_n * nb
        p4 = k_y * (nb + sb)
        p5 = gamma_y * y
        
        a0 = p0 + p1 + p2 + p3 + p4 + p5
        
        if a0 <= 0:
            break
            
        # Draw random numbers
        r1 = np.random.rand()
        r2 = np.random.rand()
        
        tau = -np.log(r1) / a0
        t += tau
        
        val = r2 * a0
        
        # Execute the sampled reaction
        if val < p0:
            state[1] -= 1  # sf
            state[3] += 1  # sb
        elif val < p0 + p1:
            state[1] += 1  # sf
            state[3] -= 1  # sb
        elif val < p0 + p1 + p2:
            state[0] -= 1  # nf
            state[2] += 1  # nb
        elif val < p0 + p1 + p2 + p3:
            state[0] += 1  # nf
            state[2] -= 1  # nb
        elif val < p0 + p1 + p2 + p3 + p4:
            state[4] += 1  # y
        else:
            state[4] -= 1  # y
            
    return state[4] # return final mRNA count

@njit
def fast_simulator_wrapper(alpha_s, beta_s, alpha_n, beta_n, k_y, gamma_y, t_max, num_cells):
    results = np.empty(num_cells)
    for i in range(num_cells):
        results[i] = fast_ssa_monomer(alpha_s, beta_s, alpha_n, beta_n, k_y, gamma_y, t_max)
    return np.sort(results)

def pymc_fast_simulator(rng, alpha_s, beta_s, alpha_n, beta_n, k_y, gamma_y, size=None):
    """
    Wrapper to bridge PyMC parameters with the Numba function.
    """
    a_s = np.asarray(alpha_s).item()
    b_s = np.asarray(beta_s).item()
    a_n = np.asarray(alpha_n).item()
    b_n = np.asarray(beta_n).item()
    g_y = np.asarray(gamma_y).item()
    k_y = np.asarray(k_y).item()

    return fast_simulator_wrapper(
        a_s, 
        b_s, 
        a_n,
        b_n,
        k_y,
        g_y,
        t_max=50.0, 
        num_cells=25 
    )
true_param = {
    "true_alpha_s": 0.5,
    "true_beta_s": 0.06,
    "true_alpha_n": 0.3,
    "true_beta_n": 0.2,
    "true_gamma_y": 1.0,
    "true_k_y": 10.0,
}
observed_data = pymc_fast_simulator(None, true_param["true_alpha_s"], 
                                    true_param["true_beta_s"],
                                    true_param["true_alpha_n"],
                                    true_param["true_beta_n"],
                                    true_param["true_gamma_y"],
                                    true_param["true_k_y"], None)


In [ ]:
with pm.Model() as fast_abc_model:
    alpha_s = pm.HalfNormal("alpha_s", sigma=5.0)
    beta_s = pm.HalfNormal("beta_s", sigma=5.0)
    alpha_n = pm.HalfNormal("alpha_n", sigma=5.0)    
    beta_n = pm.HalfNormal("beta_n", sigma=5.0)
    k_y = pm.HalfNormal("k_y", sigma=5.0)
    gamma_y = pm.HalfNormal("gamma_y", sigma=5.0)

    
    sim = pm.Simulator(
        "sim", 
        pymc_fast_simulator, 
        params=[alpha_s, beta_s, alpha_n, beta_n, k_y, gamma_y], 
        epsilon=1.0,
        sum_stat="sort", 
        observed=observed_data
    )
    idata = pm.sample_smc()


Initializing SMC sampler...
/opt/miniconda3/envs/pymc_env/lib/python3.14/site-packages/pytensor/link/numba/dispatch/basic.py:234: UserWarning: Numba will use object mode to run Simulator_sim_rv{"(),(),(),(),(),()->()"}'s perform method. Set `pytensor.config.compiler_verbose = True` to see more details.
  warnings.warn(
Sampling 5 chains in 5 jobs


Output()

In [ ]:
az.plot_trace(idata)
az.summary(idata)
az.rhat(idata)
az.plot_forest(idata, ess=True)